# 02. Faster R-CNN (two-stage)

Один ноутбук для всіх запусків: `EXPERIMENT` вибирає налаштування зі словника `EXPERIMENTS`. Кожен запуск зберігає ваги, передбачення на val і test, криву навчання, метрики й аналіз етапів в окрему папку, тож порівняння робимо з файлів без перенавчання.

| Запуск | Що змінено відносно попереднього | Тип |
|---|---|---|
| `baseline` | стандартний torchvision (лише роздільність 960 і ліміт 300 рамок) | — |
| `exp1_pos_fraction` | частка об'єктів у вибірці етапу 2: 0,25 → 0,5 | гіперпараметр |
| `exp2_giou` | функція втрат рамок етапу 2: Smooth L1 → GIoU | функція втрат |

`exp2_giou` будується поверх кращого з попередніх: якщо експ. 1 не покращить результат, у ньому треба повернути `box_positive_fraction` 0,25.

`SMOKE_TEST = True` — коротка перевірка коду на кількох кадрах (на CPU).

In [1]:
EXPERIMENT = "baseline"
SMOKE_TEST = False

EXPERIMENTS = {
    "baseline": {"box_positive_fraction": 0.25, "box_loss": "smooth_l1"},
    "exp1_pos_fraction": {"box_positive_fraction": 0.5, "box_loss": "smooth_l1"},
    "exp2_giou": {"box_positive_fraction": 0.5, "box_loss": "giou"},
}
SETUP = EXPERIMENTS[EXPERIMENT]

EPOCHS = 12
BATCH = 4
LR = 0.005
IMG_SIZE = 960
MAX_DETS = 300
VAL_STEP = 5
SEED = 42

## 1. Шляхи, бібліотеки, відтворюваність

На Kaggle: дані змагання, підготовлений набір (вихід ноутбука `hw2-data`) і оцінювач `hw2_eval.py` (датасет `hw2-eval-lib`).

In [2]:
import json
import random
import subprocess
import sys
import time
from collections import defaultdict
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from torch.utils.data import DataLoader, Dataset
from torchvision.io import ImageReadMode, read_image
from torchvision.models.detection import fasterrcnn_resnet50_fpn
from torchvision.models.detection.faster_rcnn import FastRCNNPredictor

SLUG = "multiple-object-video-detection-hw-2-ucu-ai-2026"
ON_KAGGLE = Path("/kaggle/input").exists()
if ON_KAGGLE:
    INPUT = Path("/kaggle/input")
    DATA = next(p for p in (INPUT / "competitions" / SLUG, INPUT / SLUG) if p.exists())
    PREPARED = next(INPUT.glob("**/hw2_prepared/train_coco.json")).parent
    sys.path.append(str(next(INPUT.glob("**/hw2_eval.py")).parent))
    OUT = Path("/kaggle/working") / EXPERIMENT
else:
    DATA = Path.cwd().parent.parent / "hw2_data"
    PREPARED = Path.cwd().parent.parent / "hw2_prepared"
    OUT = Path.cwd().parent.parent / "hw2_runs" / EXPERIMENT

try:
    import faster_coco_eval
except ImportError:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "faster-coco-eval"], check=True)
import hw2_eval as E

OUT.mkdir(parents=True, exist_ok=True)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
GPU = torch.cuda.get_device_name(0) if DEVICE.type == "cuda" else "cpu"
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
print(GPU, DATA, PREPARED, OUT, sep="\n")

Tesla T4
/kaggle/input/competitions/multiple-object-video-detection-hw-2-ucu-ai-2026
/kaggle/input/notebooks/solomiia2006/hw2-data/hw2_prepared
/kaggle/working/baseline


## 2. Дані

Train — підготовлений набір (кожен 5-й кадр, зафарбовані зони). Val і test — оригінальні кадри. Рамки переводимо з `[x, y, w, h]` у `[x1, y1, x2, y2]`, як вимагає torchvision; класи 1–8 беремо як є (0 у torchvision — фон). Аугментація: горизонтальне віддзеркалення з імовірністю 0,5. Після кожної епохи оцінюємо кожен 5-й кадр кожного val-кліпу.

In [3]:
class FramesDataset(Dataset):
    def __init__(self, records, train=False):
        self.records = records
        self.train = train

    def __len__(self):
        return len(self.records)

    def __getitem__(self, i):
        path, image_id, boxes, labels = self.records[i]
        image = read_image(str(path), ImageReadMode.RGB).float() / 255
        boxes = boxes.clone()
        if self.train and torch.rand(1).item() < 0.5:
            image = image.flip(-1)
            boxes[:, [0, 2]] = image.shape[-1] - boxes[:, [2, 0]]
        return image, {"boxes": boxes, "labels": labels, "image_id": image_id}


def coco_records(coco, path_of):
    by_image = defaultdict(list)
    for a in coco["annotations"]:
        x, y, w, h = a["bbox"]
        if a["iscrowd"] == 0 and w >= 1 and h >= 1:
            by_image[a["image_id"]].append((x, y, x + w, y + h, a["category_id"]))
    records = []
    for info in coco["images"]:
        rows = torch.tensor(by_image[info["id"]], dtype=torch.float32).reshape(-1, 5)
        records.append((path_of(info), info["id"], rows[:, :4], rows[:, 4].long()))
    return records


def collate(batch):
    return tuple(zip(*batch))


def make_loader(records, train):
    return DataLoader(FramesDataset(records, train), batch_size=BATCH, shuffle=train, collate_fn=collate,
                      num_workers=4 if ON_KAGGLE else 0, pin_memory=DEVICE.type == "cuda",
                      generator=torch.Generator().manual_seed(SEED))

In [4]:
val_json = DATA / "val" / "annotations_coco.json"
train_coco = json.loads((PREPARED / "train_coco.json").read_text())
val_coco = json.loads(val_json.read_text())


def val_path(info):
    p = Path(info["file_name"])
    return DATA / "val" / "images" / p.parent.name / p.name


train_records = coco_records(train_coco, lambda info: PREPARED / info["file_name"])
val_records = coco_records(val_coco, val_path)

val_frames = pd.DataFrame(val_coco["images"])
val_frames["pos"] = val_frames.groupby("video")["frame_id"].rank(method="first").astype(int) - 1
val_sub_ids = set(val_frames.loc[val_frames["pos"] % VAL_STEP == 0, "id"])
val_sub_records = [r for r in val_records if r[1] in val_sub_ids]

test_frames = pd.read_csv(DATA / "test_images.csv")
test_records = [(DATA / f, i, torch.zeros((0, 4)), torch.zeros(0, dtype=torch.long))
                for f, i in zip(test_frames["file_name"], test_frames["image_id"])]

if SMOKE_TEST:
    EPOCHS = 1
    train_records, val_records, test_records = train_records[:8], val_records[:4], test_records[:4]
    val_sub_records = val_records
    val_sub_ids = {r[1] for r in val_records}

pd.Series({"train": len(train_records), "val (кожна епоха)": len(val_sub_records),
           "val (повний)": len(val_records), "test": len(test_records)})

train                4868
val (кожна епоха)     571
val (повний)         2846
test                 3811
dtype: int64

## 3. Модель

Стандартний torchvision Faster R-CNN ResNet-50 FPN з вагами COCO; останній шар замінено на 9 класів (8 + фон), як в офіційному туторіалі. Змінено лише роздільність (довша сторона 960, рішення з EDA) і ліміт рамок на кадр (300 замість 100, бо в кадрі буває до 180 об'єктів).

**Експ. 1 — частка об'єктів у вибірці етапу 2** (`box_positive_fraction`). Етап 2 вчиться на 512 пропозиціях з кадру, з них за замовчуванням не більше 25% — об'єкти. У нас у середньому 46 об'єктів на кадр, тож пропонуємо 50%. Мотивація — дисбаланс вибірки, описаний у Libra R-CNN (arXiv:1904.02701); сама зміна частки — наше простіше рішення, не їхній метод IoU-balanced sampling.

**Експ. 2 — GIoU замість Smooth L1** для рамок етапу 2 (Rezatofighi et al., arXiv:1902.09630: *"The optimal objective for a metric is the metric itself"*). torchvision рахує втрату у функції `roi_heads.fastrcnn_loss`; ми підміняємо її копією, де Smooth L1 замінено на `torchvision.ops.generalized_box_iou_loss`. Модель передбачає зсуви відносно пропозиції, тож перед GIoU розкодовуємо і передбачення, і ціль відносно однієї умовної рамки [0, 0, 1, 1]: IoU і GIoU не змінюються від зсуву й масштабування по осях, тому результат той самий, що відносно справжньої пропозиції (наше міркування). Вага втрати 10 — як у конфігурації Faster R-CNN + GIoU у MMDetection.

In [5]:
import torch.nn.functional as F
from torchvision.models.detection import roi_heads as tv_roi_heads
from torchvision.models.detection._utils import BoxCoder
from torchvision.ops import generalized_box_iou_loss

if not hasattr(tv_roi_heads, "original_fastrcnn_loss"):
    tv_roi_heads.original_fastrcnn_loss = tv_roi_heads.fastrcnn_loss
GIOU_WEIGHT = 10.0
BOX_CODER = BoxCoder((10.0, 10.0, 5.0, 5.0))


def fastrcnn_loss_giou(class_logits, box_regression, labels, regression_targets):
    labels = torch.cat(labels, dim=0)
    regression_targets = torch.cat(regression_targets, dim=0)
    classification_loss = F.cross_entropy(class_logits, labels)

    positive = torch.where(labels > 0)[0]
    box_regression = box_regression.reshape(len(labels), -1, 4)[positive, labels[positive]].float()
    reference = torch.tensor([[0.0, 0.0, 1.0, 1.0]], device=box_regression.device).expand(len(positive), 4)
    predicted = BOX_CODER.decode_single(box_regression, reference)
    target = BOX_CODER.decode_single(regression_targets[positive].float(), reference)
    box_loss = GIOU_WEIGHT * generalized_box_iou_loss(predicted, target, reduction="sum") / labels.numel()
    return classification_loss, box_loss


def build_model():
    use_giou = SETUP["box_loss"] == "giou"
    tv_roi_heads.fastrcnn_loss = fastrcnn_loss_giou if use_giou else tv_roi_heads.original_fastrcnn_loss
    model = fasterrcnn_resnet50_fpn(weights="DEFAULT", min_size=IMG_SIZE, max_size=IMG_SIZE,
                                    box_detections_per_img=MAX_DETS,
                                    box_positive_fraction=SETUP["box_positive_fraction"])
    in_features = model.roi_heads.box_predictor.cls_score.in_features
    model.roi_heads.box_predictor = FastRCNNPredictor(in_features, len(train_coco["categories"]) + 1)
    return model.to(DEVICE)

## 4. Навчання і передбачення

Як в офіційному скрипті torchvision (`references/detection`): SGD з momentum 0,9 і weight decay 1e-4, лінійний розігрів learning rate у першій епосі, змішана точність на GPU. Передбачення повертаються в пікселях оригінального кадру.

In [6]:
def to_device(targets):
    return [{k: v.to(DEVICE) if torch.is_tensor(v) else v for k, v in t.items()} for t in targets]


def train_one_epoch(model, loader, optimizer, scaler, warmup):
    model.train()
    total = 0.0
    for images, targets in loader:
        images = [im.to(DEVICE, non_blocking=True) for im in images]
        with torch.autocast(DEVICE.type, enabled=DEVICE.type == "cuda"):
            loss = sum(model(images, to_device(targets)).values())
        optimizer.zero_grad(set_to_none=True)
        scaler.scale(loss).backward()
        scale_before = scaler.get_scale()
        scaler.step(optimizer)
        scaler.update()
        step_done = scaler.get_scale() >= scale_before
        if warmup is not None and step_done:
            warmup.step()
        total += loss.item()
    return total / len(loader)


@torch.no_grad()
def predict(model, records):
    model.eval()
    detections = []
    for images, targets in make_loader(records, train=False):
        outputs = model([im.to(DEVICE) for im in images])
        for target, out in zip(targets, outputs):
            boxes = out["boxes"].cpu()
            boxes[:, 2:] -= boxes[:, :2]
            detections += [{"image_id": target["image_id"], "category_id": c, "score": round(s, 4),
                            "bbox": [round(v, 2) for v in b]}
                           for b, c, s in zip(boxes.tolist(), out["labels"].tolist(), out["scores"].tolist())]
    return detections

## 5. Навчання

Learning rate знижується в 10 разів двічі — після 16/26 і 22/26 епох, як в офіційному розкладі (26 епох, знижки на 16 і 22). Після кожної епохи — mAP на кожному 5-му кадрі val.

In [7]:
model = build_model()
optimizer = torch.optim.SGD([p for p in model.parameters() if p.requires_grad], lr=LR, momentum=0.9, weight_decay=1e-4)
milestones = [round(EPOCHS * 16 / 26), round(EPOCHS * 22 / 26)]
scheduler = torch.optim.lr_scheduler.MultiStepLR(optimizer, milestones=milestones, gamma=0.1)
scaler = torch.amp.GradScaler(DEVICE.type, enabled=DEVICE.type == "cuda")
train_loader = make_loader(train_records, train=True)
gt_val = E.load_gt(val_json)

history = []
for epoch in range(1, EPOCHS + 1):
    start = time.time()
    warmup = (torch.optim.lr_scheduler.LinearLR(optimizer, start_factor=1 / 1000,
                                                total_iters=min(1000, len(train_loader) - 1))
              if epoch == 1 else None)
    loss = train_one_epoch(model, train_loader, optimizer, scaler, warmup)
    scheduler.step()
    train_minutes = (time.time() - start) / 60
    summary, _ = E.metrics(gt_val, E.load_detections(gt_val, predict(model, val_sub_records)), val_sub_ids)
    history.append({"epoch": epoch, "loss": loss, "train_min": train_minutes, "total_min": (time.time() - start) / 60,
                    "mAP": summary["mAP"], "AP50": summary["AP50"], "AP_small": summary["AP_small"]})
    print(pd.DataFrame(history).tail(1).round(4).to_string(index=False, header=epoch == 1))

pd.DataFrame(history).to_csv(OUT / "history.csv", index=False)
torch.save(model.state_dict(), OUT / "model.pt")

Downloading: "https://download.pytorch.org/models/fasterrcnn_resnet50_fpn_coco-258fb6c6.pth" to /root/.cache/torch/hub/checkpoints/fasterrcnn_resnet50_fpn_coco-258fb6c6.pth


100%|██████████| 160M/160M [00:00<00:00, 177MB/s]


 epoch   loss  train_min  total_min    mAP   AP50  AP_small
     1 1.1837     8.0208     9.2116 0.1603 0.3435    0.0618
2 0.8839 7.8989 9.063 0.1685 0.3599 0.0595
3 0.7874 7.9015 9.0513 0.1601 0.3445 0.0728
4 0.7305 7.9099 9.0726 0.1641 0.3436 0.0746
5 0.6858 7.9183 9.0696 0.1524 0.321 0.0661
6 0.652 7.8991 9.0607 0.1477 0.3134 0.0616
7 0.6222 7.8992 9.0575 0.151 0.3277 0.0696
8 0.5541 7.8978 9.0641 0.1583 0.3258 0.0689
9 0.5413 7.9007 9.0669 0.1575 0.3225 0.0684
10 0.535 7.8924 9.0383 0.1563 0.3225 0.071
11 0.5277 7.8871 9.0607 0.1576 0.3243 0.0699
12 0.5245 7.8954 9.0533 0.1569 0.3224 0.07


## 6. Повний val, швидкість, сабміт

Головні цифри — на всіх 2 846 кадрах val через спільний оцінювач. Швидкість — медіанний час одного кадру (batch 1, на GPU, без читання файлу).

In [8]:
@torch.no_grad()
def ms_per_image(model, records, n=100):
    model.eval()
    times = []
    for path, *_ in records[:n]:
        image = (read_image(str(path), ImageReadMode.RGB).float() / 255).to(DEVICE)
        if DEVICE.type == "cuda":
            torch.cuda.synchronize()
        start = time.perf_counter()
        model([image])
        if DEVICE.type == "cuda":
            torch.cuda.synchronize()
        times.append(time.perf_counter() - start)
    return 1000 * float(np.median(times[5:] or times))


val_detections = predict(model, val_records)
summary, per_class, clips = E.report(gt_val, val_detections)
speed = ms_per_image(model, val_sub_records)

test_detections = predict(model, test_records)
E.write_submission(test_detections, DATA / "test_images.csv", OUT / "submission.csv")

config = {"experiment": EXPERIMENT, **SETUP, "gpu": GPU, "epochs": EPOCHS, "batch": BATCH, "lr": LR, "img_size": IMG_SIZE,
          "max_dets": MAX_DETS, "seed": SEED}
results = {"config": config, "val": summary.to_dict(), "per_class": per_class.to_dict(),
           "clips": clips.to_dict(), "ms_per_image": speed}
(OUT / "metrics.json").write_text(json.dumps(results, indent=1))
(OUT / "val_predictions.json").write_text(json.dumps(val_detections))
(OUT / "test_predictions.json").write_text(json.dumps(test_detections))

print(f"Швидкість: {speed:.1f} мс/кадр")
display(summary.round(4), per_class.round(4), clips.round(4))

Швидкість: 109.1 мс/кадр


mAP              0.1565
AP50             0.3203
AP75             0.1374
AP_small         0.0691
AP_medium        0.1421
AP_large         0.3124
mAP_test_like    0.1717
dtype: float64

person      0.2252
bicycle     0.1017
car         0.3367
van         0.2051
truck       0.1581
tricycle    0.1533
bus         0.0009
motor       0.0711
dtype: float64

uav0000086_00000_v    0.1217
uav0000117_02622_v    0.1858
uav0000137_00458_v    0.2292
uav0000182_00000_v    0.1332
uav0000268_05773_v    0.0902
uav0000305_00000_v    0.1970
uav0000339_00001_v    0.2179
dtype: float64

## 7. Внесок кожного етапу

На кожному 5-му кадрі val для кожного розміченого об'єкта: чи накрила його хоч одна пропозиція RPN (IoU ≥ 0,5) і чи знайшла його фінальна рамка будь-якого класу; середній IoU найкращої пропозиції і найкращої фінальної рамки — окремо для small / medium / large. Якщо RPN пропустив об'єкт, етап 2 його вже не знайде, тож recall пропозицій — «стеля» моделі.

Пропозиції отримуємо, викликаючи етапи моделі окремо (`transform` → `backbone` → `rpn`), як у `GeneralizedRCNN.forward` у torchvision, і переводимо в пікселі оригінального кадру функцією `resize_boxes` з `torchvision.models.detection.transform`.

In [9]:
from torchvision.models.detection.transform import resize_boxes
from torchvision.ops import box_iou


@torch.no_grad()
def stage_overlaps(model, records, detections):
    model.eval()
    final_boxes = defaultdict(list)
    for d in detections:
        x, y, w, h = d["bbox"]
        final_boxes[d["image_id"]].append((x, y, x + w, y + h))
    rows = []
    for images, targets in make_loader(records, train=False):
        images = [im.to(DEVICE) for im in images]
        original_sizes = [tuple(im.shape[-2:]) for im in images]
        image_list, _ = model.transform(images)
        proposals, _ = model.rpn(image_list, model.backbone(image_list.tensors))
        for target, props, resized, original in zip(targets, proposals, image_list.image_sizes, original_sizes):
            gt = target["boxes"]
            if len(gt) == 0:
                continue
            props = resize_boxes(props, resized, original).cpu()
            final = torch.tensor(final_boxes[target["image_id"]], dtype=torch.float32).reshape(-1, 4)
            rows.append(pd.DataFrame({
                "frame": target["image_id"],
                "area": ((gt[:, 2] - gt[:, 0]) * (gt[:, 3] - gt[:, 1])).numpy(),
                "best_proposal_iou": box_iou(gt, props).max(dim=1).values.numpy() if len(props) else 0.0,
                "best_final_iou": box_iou(gt, final).max(dim=1).values.numpy() if len(final) else 0.0,
                "proposals": len(props),
                "final": len(final),
            }))
    return pd.concat(rows, ignore_index=True)


def stage_summary(group):
    return pd.Series({
        "recall пропозицій RPN (IoU ≥ 0,5)": (group["best_proposal_iou"] >= 0.5).mean(),
        "recall фінальних рамок (IoU ≥ 0,5)": (group["best_final_iou"] >= 0.5).mean(),
        "середній IoU найкращої пропозиції": group["best_proposal_iou"].mean(),
        "середній IoU найкращої фінальної рамки": group["best_final_iou"].mean(),
    })


overlaps = stage_overlaps(model, val_sub_records, [d for d in val_detections if d["image_id"] in val_sub_ids])
overlaps["size"] = pd.cut(overlaps["area"], [0, 32**2, 96**2, np.inf], labels=["small", "medium", "large"], right=False)
stages = pd.concat({"all": stage_summary(overlaps),
                    **{size: stage_summary(g) for size, g in overlaps.groupby("size", observed=True)}}, axis=1)
stages.to_csv(OUT / "stages.csv")

per_frame = overlaps.drop_duplicates("frame")[["proposals", "final"]].median()
print(f"Рамок на кадр (медіана): пропозицій RPN {per_frame['proposals']:.0f}, фінальних {per_frame['final']:.0f}")
display(stages.round(3))

Рамок на кадр (медіана): пропозицій RPN 1000, фінальних 66


,all,small,medium,large
"recall пропозицій RPN (IoU ≥ 0,5)",0.724,0.546,0.731,0.981
"recall фінальних рамок (IoU ≥ 0,5)",0.678,0.496,0.684,0.948
середній IoU найкращої пропозиції,0.590,0.454,0.592,0.810
середній IoU найкращої фінальної рамки,0.546,0.409,0.545,0.790


## Джерела

**Бібліотеки**
- PyTorch і torchvision — реалізація Faster R-CNN ResNet-50 FPN і ваги, попередньо навчені на COCO (`fasterrcnn_resnet50_fpn(weights="DEFAULT")`); `generalized_box_iou_loss`, `box_iou`, `BoxCoder`: https://docs.pytorch.org/vision/main/models.html
- faster-coco-eval (https://github.com/MiXaiLL76/faster_coco_eval) — COCO-метрики через спільний оцінювач `hw2_eval.py`.

**Код, взятий або адаптований з torchvision**
- Заміна останнього шару на свою кількість класів (`FastRCNNPredictor`) — TorchVision Object Detection Finetuning Tutorial: https://docs.pytorch.org/tutorials/intermediate/torchvision_tutorial.html
- Рецепт навчання: SGD (momentum 0,9, weight decay 1e-4), learning rate 0,02 для batch 16 (у нас перераховано на batch 4), знижки lr на 16-й і 22-й з 26 епох (у нас у тих самих пропорціях), горизонтальне віддзеркалення, змішана точність — `references/detection/train.py`: https://github.com/pytorch/vision/tree/main/references/detection
- Лінійний розігрів learning rate у першій епосі (`LinearLR`, `start_factor=1/1000`, до 1000 кроків) — `references/detection/engine.py`, там само.
- `fastrcnn_loss_giou` — копія `torchvision/models/detection/roi_heads.py::fastrcnn_loss`, у якій Smooth L1 (`beta=1/9`) замінено на GIoU; ваги кодування рамок (10, 10, 5, 5) — з `RoIHeads` там само. Розкодування відносно умовної рамки [0, 0, 1, 1] — наше рішення.
- Виклик етапів моделі окремо (`transform` → `backbone` → `rpn`) — за `torchvision/models/detection/generalized_rcnn.py::GeneralizedRCNN.forward`; переведення пропозицій у пікселі оригіналу — `transform.resize_boxes`.

**Інші реалізації**
- Вага GIoU-втрати 10 — конфігурація MMDetection `configs/faster_rcnn/faster-rcnn_r50_fpn_giou_1x_coco.py` (`loss_bbox=dict(type='GIoULoss', loss_weight=10.0)`): https://github.com/open-mmlab/mmdetection

**Статті**
- Faster R-CNN — Ren et al., 2015, arXiv:1506.01497 (двоетапна схема: RPN + RoI head).
- Feature Pyramid Networks — Lin et al., 2017, arXiv:1612.03144 (FPN у backbone).
- Libra R-CNN — Pang et al., 2019, arXiv:1904.02701 (дисбаланс на рівні вибірки — мотивація експ. 1).
- GIoU — Rezatofighi et al., 2019, arXiv:1902.09630 (функція втрат експ. 2).
- COCO detection metrics — https://cocodataset.org/#detection-eval

**Наші рішення (не з джерел):** роздільність 960 і ліміт 300 рамок на кадр (з EDA, ноутбук `01_data`), перерахунок learning rate під batch 4, оцінка кожного 5-го val-кадру після кожної епохи, значення 0,5 для частки об'єктів у вибірці (експ. 1), метрики внеску етапів (recall пропозицій і середній IoU найкращої рамки по розмірах).